In [0]:
%run /Users/arnaldo.jbj@gmail.com/ElectricalGeneratedDistribution-Project/pipelines/02_extracting_from_bronze_to_treatments_into_silver/step00_IMPORTS

In [0]:
%run /Users/arnaldo.jbj@gmail.com/ElectricalGeneratedDistribution-Project/pipelines/02_extracting_from_bronze_to_treatments_into_silver/step00_AUXILIARY_FUNCTIONS

In [0]:
TABLE_ORIGIN = 'bandeiras_tarifarias'

TABLE_DESTINY = 'tax_flags'

df = (
        spark.read.table(f"bronze.{TABLE_ORIGIN}")
             .withColumnsRenamed(
                                  {
                                    'mes_referencia' : 'month',
                                    'bandeira' : 'flag',
                                    'adicional_rs_kwh' : 'additional_charge_kwh',
                                  }
                                )
             .withColumn('additional_charge_kwh', F.col('additional_charge_kwh').cast('float'))
     )

In [0]:
df.printSchema()

In [0]:
check_duplicates(df,'month')

In [0]:
display(null_or_blank(df))

In [0]:
# Incremental load: never replaces silver.tax_flags — appends only new reference months
target = f"silver.{TABLE_DESTINY}"
key_cols = ['month']  # natural key: one flag per reference month

if not spark.catalog.tableExists(target):
    # first load: table does not exist yet — create it
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target)
    print(f"✓ {target} created ({df.count()} rows)")
else:
    # append only records whose key is not present in silver yet
    df_new = df.join(spark.table(target).select(*key_cols).distinct(), on=key_cols, how='left_anti')
    n_new = df_new.count()
    if n_new:
        df_new.write.format("delta").mode("append").saveAsTable(target)
        print(f"✓ {target} appended {n_new} new row(s)")
    else:
        print(f"↷ {target}: no new records — nothing appended")